<a href="https://colab.research.google.com/github/takakei510/tetra-peg-network/blob/main/notebooks/H2_report_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# H₂報告図の再作図
Driveの集計CSVから図を作ります。描画コードはGitHubで管理し、指定コミットから取得します。大規模原データは読みません。CPUで十分です。
Driveの `MyDrive/tetra-peg-network/h2_report` に必要なのは `size_relation.csv` だけです。


## 報告での位置づけと選定理由
先に [代表条件の選定根拠ノート](https://colab.research.google.com/github/takakei510/tetra-peg-network/blob/main/notebooks/Selection_basis_colab.ipynb) で全占有率のPmaxとSfinを示します。
L=128について、0.41（ピークより低い側で平均Pmax≤5%となる最大の占有率）、0.43（平均Sfinが最大の測定点）、0.60（ピークより高い側で平均Pmax≥80%となる最小の占有率）を比較します。5%・80%は操作的な比較基準で、0.43は厳密な転移点ではありません。
このノートはDriveの既存size_relation.csvを使います。全占有率の集計済みCSVなので再シミュレーションは不要です。新しい図はfigures_selected_colabに保存し、旧条件のfigures_colabとは分けます。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## データ・描画条件・コードの版
Driveフォルダが異なる場合はROOTを変更します。LとPHISで描画条件を指定します。
CODE_REFは描画コードを固定するコミット番号です。コード更新時はこの値も更新します。`main` を指定して常に最新版を取得することもできますが、再現には表示されたコミット番号を使ってください。


In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/tetra-peg-network/h2_report")
L = 128
PHIS = [0.41, 0.43, 0.60]
CODE_REF = "9232785694fcf4c52ca3c698ba10e07fd2fac757"
REPOSITORY = "takakei510/tetra-peg-network"
OUTPUT = ROOT / "figures_selected_colab"
assert (ROOT / "size_relation.csv").is_file(), "ROOTを実際の配置先に変更してください"
print("入力:", ROOT / "size_relation.csv")


## GitHubから描画コードを取得
指定した版のコードをColabの一時領域に取得します。Driveに置いた古い `plot_report.py` は使用しません。取得に失敗した場合は停止します。


In [ ]:
import json
import urllib.request

api_url = f"https://api.github.com/repos/{REPOSITORY}/commits/{CODE_REF}"
request = urllib.request.Request(api_url, headers={"Accept": "application/vnd.github+json"})
with urllib.request.urlopen(request, timeout=60) as response:
    CODE_COMMIT = json.load(response)["sha"]
if len(CODE_COMMIT) != 40 or any(c not in "0123456789abcdef" for c in CODE_COMMIT):
    raise ValueError("GitHubから有効なコミット番号を取得できませんでした")
script_url = f"https://raw.githubusercontent.com/{REPOSITORY}/{CODE_COMMIT}/scripts/analysis/plot_report.py"
with urllib.request.urlopen(script_url, timeout=60) as response:
    script_bytes = response.read()
compile(script_bytes.decode("utf-8"), script_url, "exec")
CODE_DIR = Path("/content/h2_report_code") / CODE_COMMIT
CODE_DIR.mkdir(parents=True, exist_ok=True)
SCRIPT = CODE_DIR / "plot_report.py"
SCRIPT.write_bytes(script_bytes)
print("描画コードのコミット:", CODE_COMMIT)
print("描画コード:", SCRIPT)


## 日本語フォント
Colabの実行環境に導入します。ノートPCへのインストールではありません。

In [ ]:
import subprocess
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-cjk"], check=True)


## 図を出力
データは読み取りのみ。図はDrive内のfigures_selected_colabへ保存し、既存の同名図を更新します。
入力CSVと描画コードのSHA-256、コードのコミット番号、描画条件とライブラリ版を `run_metadata.json` に保存します。


In [ ]:
import sys
import hashlib
import platform
import matplotlib
from datetime import datetime, timezone

command = [sys.executable, str(SCRIPT), "--input", str(ROOT / "size_relation.csv"),
           "--output", str(OUTPUT), "--L", str(L), "--phi", *map(str, PHIS)]
subprocess.run(command, check=True)
metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repository": REPOSITORY, "code_ref": CODE_REF, "code_commit": CODE_COMMIT,
    "script_sha256": hashlib.sha256(SCRIPT.read_bytes()).hexdigest(),
    "input": str(ROOT / "size_relation.csv"),
    "input_sha256": hashlib.sha256((ROOT / "size_relation.csv").read_bytes()).hexdigest(),
    "L": L, "phi": PHIS, "python": platform.python_version(),
    "matplotlib": matplotlib.__version__, "command": command,
}
(OUTPUT / "run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("実行記録:", OUTPUT / "run_metadata.json")


In [ ]:
from IPython.display import display, Image
for name in ["count_log", "count_linear", "h2_fraction"]:
    display(Image(filename=str(OUTPUT / (name + ".png"))))


## 読み方と限界
横軸は等間隔に並べたサイズ区間です。連続量の通常・対数軸ではありません。
クラスタ数は全100試料で平均、H₂割合は区間のある試料だけで平均。H₂=0のクラスタも含みます。
色付き点はn≥50、誤差棒SE。白抜きはn<50。SEは試料間の平均の不確かさです。
代表3条件だけで全φへの一般化やH₂の因果効果は主張しません。Colabの一時領域は永続保存されないため、図はDriveへ保存しています。

## 結果を説明する前の確認
- H2末端割合は、クラスタ内でH2ペアに使われた末端数 / そのクラスタの全末端数です。クラスタ数の割合とは違います。
- H2図は各試料の最大クラスタを1個除いています。一方、個数の図は最大クラスタを含みます。
- 0.60では65〜128分子区間の寄与試料数が6と少なく、129〜256分子区間は1です。色付き点と同じ確かさでは比較できません。
- 0.60の33〜64分子区間などに増加もあるため、全区間で単調減少すると説明しません。
- クラスタサイズとH2末端割合の関連から、H2が成長を抑制したという因果までは判断できません。
